## Load data and merge database

In [2]:
import pandas as pd
# Temperature dataset
tx_path = "/Users/mohammadrezanilchiyan/Desktop/UE/Datascientest/project/Temperature/Data/processed_2025.12.15/Marseille_daily_TX_raw.csv"

# Wind dataset
wind_path = "/Users/mohammadrezanilchiyan/Desktop/UE/Datascientest/project/Temperature/Data/processed_2025.12.15/temp_wind_marseille.csv"


In [3]:
# Temperature
df_tx = pd.read_csv(tx_path, parse_dates=["date"])

# Wind
df_wind = pd.read_csv(wind_path, parse_dates=["date"])


In [4]:
# Temperature dataset: keep Station name (NOM_USUEL) and TX
df_tx = df_tx[["date", "TX", "NOM_USUEL"]]

# Wind dataset: keep only rows where main features exist
df_wind = df_wind.dropna(subset=["wind_max_inst_ms", "temp_max_c", "wind_dir_inst_deg"])

# Rename columns for clarity
df_wind = df_wind.rename(
    columns={
        "wind_max_inst_ms": "Wx",     # max instantaneous wind
        "temp_max_c": "Tx",           # max temperature
        "wind_dir_inst_deg": "Wx_dir" # wind direction
    }
)


In [5]:
df_merged = (
    df_tx.merge(df_wind, on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)


In [8]:
df_merged[["date", "NOM_USUEL", "TX", "Tx", "Wx", "Wx_dir"]].head(7)


,date,NOM_USUEL,TX,Tx,Wx,Wx_dir
0,1950-01-01,MARIGNANE,11.4,11.4,4.0,290.0
1,1950-01-01,ISTRES,10.5,11.4,4.0,290.0
2,1950-01-01,SALON DE PROVENCE,9.8,11.4,4.0,290.0
3,1950-01-01,BEC DE L AIGLE,13.6,11.4,4.0,290.0
4,1950-01-02,MARIGNANE,9.0,9.0,9.0,320.0
5,1950-01-02,SALON DE PROVENCE,8.0,9.0,9.0,320.0
6,1950-01-02,BEC DE L AIGLE,12.0,9.0,9.0,320.0


##  Predictor features preparation

In [7]:
# wind direction encoding
import numpy as np

df_merged["Wx_dir_rad"] = np.deg2rad(df_merged["Wx_dir"])
df_merged["Wx_dir_sin"] = np.sin(df_merged["Wx_dir_rad"])
df_merged["Wx_dir_cos"] = np.cos(df_merged["Wx_dir_rad"])

# Modeling 

In [10]:
#checks, for each day, Is the daily maximum temperature (TX) at least 35°C? , by end
#Convert True / False to numbers. True (1) and False(0)
df_merged["extreme_heat"] = (df_merged["TX"] >= 35).astype(int)

In [11]:
threshold = df_merged["TX"].quantile(0.99)
df_merged["extreme_heat"] = (df_merged["TX"] >= threshold).astype(int)

# first try(Logistic regression)

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [13]:
features = ["Wx", "Wx_dir_sin", "Wx_dir_cos"]
X = df_merged[features]
y = df_merged["extreme_heat"]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False)

In [15]:
model = LogisticRegression()
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [16]:
y_pred = model.predict(X_test)

In [18]:
#how likely each day is an extreme heat day
#The model already learned from the 0/1 target — now it calculates the probability of 1 for each new sample.
risk_prob = model.predict_proba(X_test)[:, 1]

In [19]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, risk_prob)
print("ROC-AUC:", auc)

ROC-AUC: 0.6638650809901507


In [20]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.99    101505
           1       0.00      0.00      0.00      2723

    accuracy                           0.97    104228
   macro avg       0.49      0.50      0.49    104228
weighted avg       0.95      0.97      0.96    104228



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


support = number of true instances of each class in the test set.
Class 0 (normal day) 101,505 samples
Class 1 (extreme heat)  2,723 samples

Precision , Class 0: 97% of predicted normal days are correct
Class 1: 0.00 , none of the predicted extreme heat days are correct

Recall , Class 0: 100% of actual normal days are correctly predicted
Class 1:  0% of extreme heat days are detected
This confirms the problem: your model is not detecting any extreme heat events.

In [3]:
## the model Failed 

## fix/improve the model

- Handle imbalance:Oversample extreme heat days (SMOTE, RandomOverSampler)

- Try more powerful classifiers:
Random Forest, Gradient Boosting (XGBoost) handle imbalanced classes better.